# Extreme Conditions Failure Prediction — Dataset Analysis & Exploratory Notebook

This notebook performs exploratory data analysis (EDA), visualizes sensor time-series, and demonstrates an **Unsupervised Anomaly Detection (Isolation Forest)** baseline trained on the 221-sample telemetry dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Load processed dataset
data_path = '../data/processed/processed_sensor_data.csv'
if not os.path.exists(data_path):
    data_path = '../data/raw/sensor_data_2026-08-26_110040.csv'

df = pd.read_csv(data_path)
print(f"Loaded dataset shape: {df.shape}")
df.head()

## 1. Summary Statistics

In [ ]:
df.describe().T[['min', 'mean', 'max', 'std']]

## 2. Sensor Feature Time Series Plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

t_sec = (df['timestamp_ms'] - df['timestamp_ms'].iloc[0]) / 1000.0

axes[0].plot(t_sec, df['temperature_c'], label='Temp (°C)', color='red')
axes[0].set_ylabel('Temp (°C)')
axes[0].legend(loc='upper right')
axes[0].grid(True)

axes[1].plot(t_sec, df['accel_mag_g'] if 'accel_mag_g' in df.columns else df['accel_z_g'], label='Accel Mag (g)', color='orange')
axes[1].axhline(1.0, color='gray', linestyle=':')
axes[1].set_ylabel('Accel (g)')
axes[1].legend(loc='upper right')
axes[1].grid(True)

axes[2].plot(t_sec, df['gyro_mag_dps'] if 'gyro_mag_dps' in df.columns else df['gyro_z_dps'], label='Gyro Mag (dps)', color='blue')
axes[2].set_ylabel('Gyro (dps)')
axes[2].set_xlabel('Time (seconds)')
axes[2].legend(loc='upper right')
axes[2].grid(True)

plt.suptitle('Sensor Telemetry & Motion Profiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Baseline Unsupervised Anomaly Detection (Isolation Forest)

Since the dataset contains clean baseline telemetry without failure labels, we train an **Isolation Forest** model to establish an anomaly score threshold.

In [ ]:
feature_cols = [c for c in df.columns if c not in ['timestamp_ms']]
X = df[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit Isolation Forest
iso = IsolationForest(contamination=0.02, random_state=42)
df['anomaly_score'] = iso.fit_predict(X_scaled)
df['raw_score'] = iso.decision_function(X_scaled)

anomalies = df[df['anomaly_score'] == -1]
print(f"Detected {len(anomalies)} potential baseline anomalies out of {len(df)} samples.")

plt.figure(figsize=(12, 4))
plt.plot(t_sec, df['raw_score'], label='Isolation Forest Decision Score', color='purple')
plt.axhline(0.0, color='red', linestyle='--', label='Anomaly Threshold')
plt.xlabel('Time (seconds)')
plt.ylabel('Decision Score')
plt.title('Baseline Anomaly Detection Profile')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()